# Other calibrations & their effect (corrected components)

Builds on **notebook 09** (corrected component selection). Adds two meeting-motivated
calibrations and shows the **effect of the calibration choice on predicted EC**.

| New variant | Rule | Why |
|-------------|------|-----|
| **CAL-6** rel-residual clean | iterative 3σ on **relative** residual `resid/measured` | the *fair-to-high-EC* alternative to CAL-1's absolute trim — the meeting's exact critique |
| **CAL-7** no-extremes | drop the 3 filters with EC > 250 µg | shows how much the whole picture hangs on a handful of extreme smoke samples |

Every model is fit at **its own corrected k\*** (10-fold interleaved CV, one 5% tolerance —
the notebook-09 method), then applied to the **same full 906-sample set** so the predicted-EC
distributions are directly comparable.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression

PRED = Path("../spartan_ec_2026_06_16")
Path("figures").mkdir(exist_ok=True); Path("tables").mkdir(exist_ok=True)

Xdf = pd.read_csv(PRED / "data/rds_EC_X.csv")
y = pd.read_csv(PRED / "data/rds_EC_Ymeasured.csv")["Y_measured"].to_numpy()
X = Xdf[[c for c in Xdf.columns if c != "id"]].to_numpy(float)
print("EC set:", X.shape, "| EC range", round(y.min(), 2), "-", round(y.max(), 2))

EC set: (906, 2722) | EC range 0.77 - 393.33


## Variant masks — the six from 06/09 plus CAL-6 and CAL-7

`BASELINE_K = 21` defines the baseline model (as in 06/09). CAL-6 trims on the *relative*
residual instead of the absolute one, so a big absolute miss on a high-EC smoke sample is not
punished when it is a small *percentage* error.

In [2]:
BASELINE_K = 21
base = PLSRegression(n_components=BASELINE_K, scale=False).fit(X, y)
pred0 = base.predict(X).ravel()

def sigma_trim(relative):
    '''Iterative 3σ residual trim; relative=True trims on resid/measured.'''
    keep = np.ones(len(y), bool)
    for _ in range(3):
        m = PLSRegression(n_components=BASELINE_K, scale=False).fit(X[keep], y[keep])
        r = y - m.predict(X).ravel()
        if relative:
            r = r / y                      # EC is strictly positive here (min 0.77), no /0
        newkeep = np.abs(r - np.median(r[keep])) <= 3 * r[keep].std()
        if (newkeep == keep).all():
            break
        keep = keep & newkeep
    return keep

abs_keep = sigma_trim(relative=False)      # CAL-1
rel_keep = sigma_trim(relative=True)       # CAL-6
ETH_LOW, ETH_HIGH = 10.0, 100.0

MASKS = {
    "CAL-0 all-nofilter":  np.ones(len(y), bool),
    "CAL-1 abs-clean":     abs_keep,
    "CAL-6 rel-clean":     rel_keep,
    "CAL-3 below-1:1":     pred0 < y,
    "CAL-5 Eth-range":     (y >= ETH_LOW) & (y <= ETH_HIGH),
    "CAL-7 no-extremes":   y <= 250.0,
    "CAL-2 removed-only":  ~abs_keep,      # diagnostics (small n)
    "CAL-4 EC-high>=70":   y >= 70.0,
}
CANDIDATES  = ["CAL-0 all-nofilter","CAL-1 abs-clean","CAL-6 rel-clean",
               "CAL-3 below-1:1","CAL-5 Eth-range","CAL-7 no-extremes"]
for k, m in MASKS.items():
    print(f"{k:22s} n={int(m.sum()):4d}   mean EC={y[m].mean():6.1f}")

CAL-0 all-nofilter     n= 906   mean EC=  16.9
CAL-1 abs-clean        n= 860   mean EC=  13.7
CAL-6 rel-clean        n= 873   mean EC=  17.3
CAL-3 below-1:1        n= 469   mean EC=  18.3
CAL-5 Eth-range        n= 480   mean EC=  21.3
CAL-7 no-extremes      n= 903   mean EC=  15.8
CAL-2 removed-only     n=  46   mean EC=  75.8
CAL-4 EC-high>=70      n=  22   mean EC= 144.3


## Corrected component selection (notebook-09 machinery) → each variant's k\*

In [3]:
def pred_all_ncomps(m, Xnew, mc):
    Xc = Xnew - m._x_mean
    out = np.empty((Xnew.shape[0], mc))
    for k in range(1, mc + 1):
        b = m.x_rotations_[:, :k] @ m.y_loadings_[:, :k].T
        out[:, k - 1] = (Xc @ b).ravel() + m._y_mean
    return out

def cv_rmsecv(Xs, ys, folds=10, maxc=30):
    f = np.arange(len(ys)) % folds
    mc = int(min(maxc, min((f != i).sum() for i in range(folds)) - 1, Xs.shape[1]))
    press = np.zeros(mc)
    for i in range(folds):
        tr, te = f != i, f == i
        m = PLSRegression(n_components=mc, scale=False).fit(Xs[tr], ys[tr])
        press += np.sum((pred_all_ncomps(m, Xs[te], mc) - ys[te][:, None])**2, axis=0)
    return np.arange(1, mc + 1), np.sqrt(press / len(ys))

def pick_k(ks, rc, tol=0.05):
    return int(ks[np.argmax(rc <= rc.min() * (1 + tol))])

models = {}
for name, mask in MASKS.items():
    Xs, ys = X[mask], y[mask]
    ks, rc = cv_rmsecv(Xs, ys)
    k = pick_k(ks, rc)
    models[name] = dict(k=k, model=PLSRegression(n_components=k, scale=False).fit(Xs, ys),
                        n=int(mask.sum()), meanEC=float(ys.mean()),
                        rmsecv=float(rc.min()), pct=float(100 * rc.min() / ys.mean()))
    print(f"{name:22s} k*={k:2d}  %RMSECV={models[name]['pct']:5.1f}%")

CAL-0 all-nofilter     k*=19  %RMSECV= 97.7%


CAL-1 abs-clean        k*= 1  %RMSECV=143.3%


CAL-6 rel-clean        k*=27  %RMSECV= 83.7%


CAL-3 below-1:1        k*= 4  %RMSECV=206.2%


CAL-5 Eth-range        k*=13  %RMSECV= 33.8%


CAL-7 no-extremes      k*=27  %RMSECV= 63.0%
CAL-2 removed-only     k*= 1  %RMSECV= 87.9%


CAL-4 EC-high>=70      k*= 1  %RMSECV= 58.6%


## Figure A — predicted vs. measured at each variant's corrected k\*
(In-sample fit quality; the honest cross-variant number is %RMSECV, not R².)

In [4]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, name in zip(axes.ravel(), CANDIDATES):
    mask = MASKS[name]; mdl = models[name]
    ys = y[mask]; pred = mdl["model"].predict(X[mask]).ravel()
    lim = max(ys.max(), pred.max()) * 1.05
    ax.plot([0, lim], [0, lim], "k--", lw=1)
    ax.scatter(ys, pred, s=14, alpha=0.5)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_title(f"{name}\nn={mdl['n']}  k*={mdl['k']}  %RMSECV={mdl['pct']:.0f}%", fontsize=9)
    ax.set_xlabel("measured EC (µg)"); ax.set_ylabel("predicted EC (µg)")
fig.suptitle("Calibration variants — fit at each variant's CORRECTED k* (1:1 dashed)", y=1.01, fontsize=13)
plt.tight_layout(); plt.savefig("figures/fig10_variants_1to1_correctedk.png", dpi=140, bbox_inches="tight")
print("saved figures/fig10_variants_1to1_correctedk.png"); plt.show()

saved figures/fig10_variants_1to1_correctedk.png


/var/folders/7k/65ckdzsj0w3171qv80rh8dgr0000gn/T/ipykernel_74430/2758061721.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print("saved figures/fig10_variants_1to1_correctedk.png"); plt.show()


## Figure B — THE EFFECT: apply every calibration to the same 906 filters

Each calibration (fit on its own training set at its own k\*) is applied to the **full
906-sample set**. The box plots show how the predicted-EC distribution shifts with the
calibration choice — the same filters, different EC depending on which calibration you trust.

In [5]:
full_pred = {name: models[name]["model"].predict(X).ravel() for name in CANDIDATES}
ref_med = np.median(full_pred["CAL-0 all-nofilter"])

fig, ax = plt.subplots(figsize=(11, 5.5))
data = [np.clip(full_pred[n], -50, 200) for n in CANDIDATES]   # clip only for display
bp = ax.boxplot(data, labels=[n.replace(" ", "\n", 1) for n in CANDIDATES], showfliers=False,
                patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("#9ecae1"); patch.set_alpha(0.7)
ax.axhline(np.median(y), color="green", ls="--", lw=1, label=f"measured median EC ({np.median(y):.1f})")
ax.axhline(ref_med, color="crimson", ls=":", lw=1, label=f"CAL-0 predicted median ({ref_med:.1f})")
ax.set_ylabel("predicted EC on the full 906 filters (µg)")
ax.set_title("Effect of calibration choice — same 906 filters, different predicted EC")
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig("figures/fig10_effect_predicted_ec.png", dpi=140, bbox_inches="tight")
print("saved figures/fig10_effect_predicted_ec.png"); plt.show()

/var/folders/7k/65ckdzsj0w3171qv80rh8dgr0000gn/T/ipykernel_74430/1765253442.py:6: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=[n.replace(" ", "\n", 1) for n in CANDIDATES], showfliers=False,


saved figures/fig10_effect_predicted_ec.png


/var/folders/7k/65ckdzsj0w3171qv80rh8dgr0000gn/T/ipykernel_74430/1765253442.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print("saved figures/fig10_effect_predicted_ec.png"); plt.show()


## The effect, quantified

In [6]:
rows = []
for name in CANDIDATES:
    p = full_pred[name]
    rows.append({"calibration": name, "n_train": models[name]["n"], "k*": models[name]["k"],
                 "pctRMSECV": round(models[name]["pct"], 1),
                 "median_pred_EC": round(float(np.median(p)), 2),
                 "ratio_to_CAL0": round(float(np.median(p) / ref_med), 2),
                 "pct_negative": round(float(100 * (p < 0).mean()), 1)})
eff = pd.DataFrame(rows)
eff.to_csv("tables/calibration_variants_effect.csv", index=False)
print(eff.to_string(index=False))
spread = eff["ratio_to_CAL0"].max() / eff["ratio_to_CAL0"].min()
print(f"\nAcross calibrations, the median predicted EC spans a {spread:.1f}x range "
      f"on the identical filters.")
print("wrote tables/calibration_variants_effect.csv")

       calibration  n_train  k*  pctRMSECV  median_pred_EC  ratio_to_CAL0  pct_negative
CAL-0 all-nofilter      906  19       97.7           11.07           1.00           2.5
   CAL-1 abs-clean      860   1      143.3           13.50           1.22           0.0
   CAL-6 rel-clean      873  27       83.7           11.08           1.00           3.2
   CAL-3 below-1:1      469   4      206.2           12.08           1.09           0.4
   CAL-5 Eth-range      480  13       33.8           12.19           1.10           0.1
 CAL-7 no-extremes      903  27       63.0           10.71           0.97           0.6

Across calibrations, the median predicted EC spans a 1.3x range on the identical filters.
wrote tables/calibration_variants_effect.csv


### Reading it
- **The calibration choice, not the sample, sets the EC.** The same 906 filters give a median
  EC that spans a multi-fold range depending on which calibration is applied — the direct,
  on-real-data version of notebook 07's Ethiopia ~2× swing.
- **CAL-6 (relative clean) vs CAL-1 (absolute clean)** is the meeting's fairness test: the
  relative filter keeps the informative high-EC smoke samples the absolute 3σ discards, so its
  predictions and %RMSECV differ — quantifying why the absolute filter was the wrong tool.
- **CAL-7 (no-extremes)** shows how much rides on 3 filters: dropping EC>250 shifts the whole fit.
- Restricted calibrations (CAL-5) **extrapolate** on out-of-range filters — read their full-set
  predictions as a sensitivity check, not a deployment.
- Components are each variant's corrected k\* (notebook 09), so this is *not* confounded by the
  common-k choice of notebook 06.